In [2]:
import h5py
import os
import pandas as pd
import pathlib
import pickle
import scipy.stats as stats
import soxr
#! change below to spatial_attn_lighting if want to use with modular 
from src.spatial_attn_lightning import BinauralAttentionModule 
# from corpus.swc_mono_test import SWCMonoTestSetH5Dataset, SWCMonoTestSetH5DatasetForUnitTuning
import corpus.swc_mono_test as swc_mono_test
import importlib
import src.audio_transforms as at
import torch
import yaml
import numpy as np
import torch
from tqdm.auto import tqdm
from datetime import datetime
import sys 
import torchaudio.transforms as T

sys.path.append('../')
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
# torch._dynamo.config.skip_nnmodule_hook_guards=False

In [3]:
config_path = "config/binaural_attn/word_task_v10_main_feature_gain_config.yaml"
ckpt_path = "attn_cue_models/word_task_v10_main_feature_gain_config/checkpoints/epoch=1-step=24679.ckpt"
# old_style = True 

config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)
# config['model']['new_modulee']=Tr
# config['getting_acts']  = True
# model = BinauralAttentionModule.load_from_checkpoint(checkpoint_path=ckpt_path, config=config, strict=False).cuda()


In [4]:
model_name = pathlib.Path(config_path).stem
model_name

'word_task_v10_main_feature_gain_config'

In [5]:
### Dev new test set

In [11]:
importlib.reload(swc_mono_test)

SWCMonoTestSetH5DatasetForUnitTuning = swc_mono_test.SWCMonoTestSetH5DatasetForUnitTuning

# coch_gram = model.coch_gram.cuda()

snr=0
audio_transforms = at.AudioCompose([
            at.AudioToTensor(),
            at.CombineWithRandomDBSNR(low_snr=snr, high_snr=snr), 
            at.RMSNormalizeForegroundAndBackground(rms_level=0.02),  # 0.02 is the default for CV-based models 
            at.UnsqueezeAudio(dim=0),
            at.DuplicateChannel()
            ])

stim_path = "/om/user/imgriff/datasets/human_word_rec_SWC_2024/model_eval_stim.h5"

condition = "one_distractor" 
dataset = SWCMonoTestSetH5DatasetForUnitTuning(h5_path=stim_path,
                                model_sr=config['audio']['rep_kwargs']['sr'],
                               )

In [7]:
import librosa 


def get_avg_f0(sig, SR, fmin=80, fmax=300):
    """
    Get average f0 of a signal using librosa's pyin function.
    """
    # get talker f0 - bound to range of human voice 
    target_f0, voice_flag, voice_probs = librosa.pyin(sig, fmin=80, fmax=300, sr=SR, fill_na=np.nan)
    target_f0 = target_f0[voice_flag]        
    avg_target_f0 = np.nanmean(target_f0)
    return avg_target_f0

In [12]:
batch =dataset[0]

In [13]:
batch

(array([0., 0., 0., ..., 0., 0., 0.], dtype=float32),
 array([0., 0., 0., ..., 0., 0., 0.], dtype=float32),
 array([0., 0., 0., ..., 0., 0., 0.], dtype=float32),
 array([0., 0., 0., ..., 0., 0., 0.], dtype=float32),
 array([-0.01948547, -0.02210995, -0.02082823, ...,  0.01808111,
         0.01710457,  0.01563974], dtype=float32),
 0,
 191.97674170199025,
 227.63830756781104,
 138.53299150797477)

In [6]:
conv_modules = {name:module for name, module in model.model._orig_mod.model_dict.named_children() if 'conv' in name or 'relu' in name}
# add relu fc 
relu_fc = model.model._orig_mod.relufc
modules = {**conv_modules, **{'relufc': relu_fc}}

In [18]:
h5_file = h5py.File(stim_path, 'r') # modules
print(h5_file.keys())

<KeysViewHDF5 ['1-talker-dutch-different', '1-talker-dutch-same', '1-talker-english-different', '1-talker-english-same', '1-talker-mandarin-different', '1-talker-mandarin-same', '2-talker', '4-talker', 'babble_dist_signal', 'cue_signal', 'diff_dist_word_int', 'music_dist_signal', 'nat_dist_signal', 'same_dist_word_int', 'ssn_dist_signal', 'target_signal', 'word_int_label']>


In [8]:
# init array to store activations
activations = {}

def get_activation(name):
    def hook(model, input, output):
        # print(name)
        if name in activations:
            activations[name] = torch.cat((activations[name], output.detach()), dim=0)
        else:
            activations[name] = output.detach()
    return hook

# dicts to store activations
# fg_reps = {}
# bg_reps = {}
# mixture_reps = {}

# register hooks 
for name, module in modules.items():

    if 'relufc' in name:
        module.register_forward_hook(get_activation(name)) 
    else:
        module[2].register_forward_hook(get_activation(name)) # [2] is relu 
    # lists to save acts in per-layer
    # fg_reps[name] = []
    # bg_reps[name] = []
    # mixture_reps[name] = []

# send model to gpu 
model = model.eval().cuda()


In [9]:
import h5py
from pathlib import Path
from scipy import stats 

In [10]:
n_activations = 1 
# get activations 

# write acts to h5 
outname = Path(f'binaural_model_attn_stage_reps/{model_name}/{model_name}_model_activations_0dB_w_cues_v2.h5')
# outname.parent.mkdir(parents=True, exist_ok=True)
# outname = 'test_act_outs.h5'
# with h5py.File(outname, 'w') as f:

with torch.no_grad():
    for ix, batch in tqdm(enumerate(dataloader),  total = n_activations):
        fg_cue, foreground, background, mixture, fg_target = batch

        # send to device
        foreground, background, mixture = foreground.cuda(), background.cuda(), mixture.cuda()
        fg_cue =  fg_cue.cuda()

        # convert to cochleagrams
        fg_cue, mixture = coch_gram(fg_cue, mixture)
        foreground, background = coch_gram(foreground, background)

        # if ix == 0:
        #         f.create_dataset('cochleagram_cue',shape=[n_activations, mixture.view(-1).shape[0]], dtype=np.float32)
        #         f.create_dataset('cochleagram_mixture', shape=[n_activations, mixture.view(-1).shape[0]], dtype=np.float32)
        #         f.create_dataset('cochleagram_fg', shape=[n_activations, mixture.view(-1).shape[0]], dtype=np.float32)
        #         f.create_dataset('cochleagram_bg', shape=[n_activations, mixture.view(-1).shape[0]], dtype=np.float32)
        # f['cochleagram_cue'][ix] = fg_cue.view(-1).cpu().numpy()
        # f['cochleagram_mixture'][ix] = mixture.view(-1).cpu().numpy()
        # f['cochleagram_fg'][ix] = foreground.view(-1).cpu().numpy()
        # f['cochleagram_bg'][ix] = background.view(-1).cpu().numpy()


        # run mixture
        pred = model(fg_cue, mixture, None)
        for layer in activations.keys():
            print(layer)
            if 'relufc' in layer:
                    mixture_acts = activations[layer] 
                    # if ix == 0:
                    # f.create_dataset(f'{layer}_mixture', shape=[n_activations, np.prod(mixture_acts.shape)], dtype=np.float32)
            else:
                cue_acts, mixture_acts = activations[layer] 
            # break 
            #     if ix == 0:
            #         f.create_dataset(f'{layer}_cue', shape=[n_activations, np.prod(cue_acts.shape)], dtype=np.float32)
            #         f.create_dataset(f'{layer}_mixture', shape=[n_activations, np.prod(mixture_acts.shape)], dtype=np.float32)
            #     f[f'{layer}_cue'][ix] = cue_acts.view(-1).cpu().numpy()
            # f[f'{layer}_mixture'][ix] = mixture_acts.view(-1).cpu().numpy()
        break
        # activations = {}
        # print(activatiions)
        
        # run fg - can skip cue 
        # pred = model(fg_cue, foreground, None)
        # for layer in activations.keys():
        #     if 'relufc' in layer:
        #             fg_acts = activations[layer] 
        #     else:
        #         _, fg_acts = activations[layer]
        #     # if ix == 0:
        #         # f.create_dataset(f'{layer}_fg', shape=[n_activations, np.prod(fg_acts.shape)], dtype=np.float32)
        #     # f[f'{layer}_fg'][ix] = fg_acts.view(-1).cpu().numpy()
        # activations = {}

        # # run bg
        # pred = model(fg_cue, background, None)
        # for layer in activations.keys():
        #     if 'relufc' in layer:
        #             bg_acts = activations[layer]
        #     else:
        #         _, bg_acts = activations[layer]
        #     # if ix == 0:
        #         # get flattened shape of bg_acts 
        #         # f.create_dataset(f'{layer}_bg', shape=[n_activations, np.prod(bg_acts.shape)], dtype=np.float32)
        #     # f[f'{layer}_bg'][ix] = bg_acts.view(-1).cpu().numpy()
        # activations = {}
        

        # Save signals without cue in forward pass 
        # run mixture
        # pred = model(None, mixture, None)
        # for layer in activations.keys():
        #     print(layer)
        #     mixture_acts = activations[layer] 
        #     break
        #     if ix == 0:
        #         f.create_dataset(f'{layer}_mixture_no_cue', shape=[n_activations, np.prod(mixture_acts.shape)], dtype=np.float32)
        #     f[f'{layer}_mixture_no_cue'][ix] = mixture_acts.view(-1).cpu().numpy()
        # # activations = {}
        # break
        # # run fg - can skip cue
        # pred = model(None, foreground, None)
        # for layer in activations.keys():
        #     fg_acts = activations[layer] 
        #     break
            # if ix == 0:
            #     f.create_dataset(f'{layer}_fg_no_cue', shape=[n_activations, np.prod(fg_acts.shape)], dtype=np.float32)
            # f[f'{layer}_fg_no_cue'][ix] = fg_acts.view(-1).cpu().numpy()
        # activations = {}

        # # run bg
        # pred = model(None, background, None)
        # for layer in activations.keys():
        #     bg_acts = activations[layer]
        #     # if ix == 0:
        #     #     f.create_dataset(f'{layer}_bg_no_cue', shape=[n_activations, np.prod(bg_acts.shape)], dtype=np.float32)
        #     # f[f'{layer}_bg_no_cue'][ix] = bg_acts.view(-1).cpu().numpy()
        # activations = {}

        # if ix == n_activations-1:
        #     break
      

  0%|          | 0/1 [00:00<?, ?it/s]

[2024-05-30 19:24:50,332] [0/0] torch._dynamo.output_graph: [WARNING] nn.Module forward/_pre hooks are only partially supported, and were detected in your model. In particular, if you do not change/remove hooks after calling .compile(), you can disregard this warning, and otherwise you may need to set torch._dynamo.config.skip_nnmodule_hook_guards=False to ensure recompiling after changing hooks.See https://pytorch.org/docs/master/compile/nn-module.html for more information and limitations. 
  0%|          | 0/1 [00:13<?, ?it/s]

conv_block_0
conv_block_1
conv_block_2
conv_block_3
conv_block_4
conv_block_5
conv_block_6
relufc


In [15]:
fg_cue.

torch.Size([1, 2, 40, 20000])

In [18]:
out_name =  Path(f'binaural_model_attn_stage_reps/{model_name}/{model_name}_layer_shape_dict.pkl')

shape_dict = {key:val.shape[1:] for key, val in activations.items()}
shape_dict['cochleagram'] = fg_cue.shape[1:]
print(shape_dict)
with open(out_name, 'wb') as f:
    pickle.dump(shape_dict, f)

{'conv_block_0': torch.Size([32, 40, 20000]), 'conv_block_1': torch.Size([64, 20, 5000]), 'conv_block_2': torch.Size([256, 10, 1250]), 'conv_block_3': torch.Size([512, 10, 250]), 'conv_block_4': torch.Size([512, 10, 63]), 'conv_block_5': torch.Size([512, 10, 63]), 'conv_block_6': torch.Size([512, 10, 63]), 'relufc': torch.Size([512]), 'cochleagram': torch.Size([2, 40, 20000])}


In [20]:
out_name

PosixPath('binaural_model_attn_stage_reps/word_task_half_co_loc_v07/word_task_half_co_loc_v07_layer_shape_dict.pkl')

In [ ]:
# h5_file = h5py.File(outname, 'r')
# print(h5_file.keys())
# # do corrs 
# layers = [key.split('_mixture')[0] for key in h5_file.keys() if 'mixture' in key]
# target_corrs = {}
# bg_corrs = {}
# for layer in layers:
#     fg_act = h5_file[f"{layer}_fg"]
#     bg_act = h5_file[f"{layer}_bg"]
#     mixture_act = h5_file[f"{layer}_mixture"]
#     n_acts = fg_act.shape[0]
#     target_corrs[layer] = [stats.pearsonr(fg_act[i], mixture_act[i])[0] for i in range(n_acts)]
#     bg_corrs[layer] = [stats.pearsonr(bg_act[i], mixture_act[i])[0] for i in range(n_acts)]

# h5_file.close()


: 

In [ ]:
import pandas as pd 

: 

In [ ]:
dfs = []
for layer in target_corrs.keys():
    df = pd.DataFrame.from_dict({'fg_corrs':target_corrs[layer],
                                 'bg_corrs':bg_corrs[layer],
                                 'layer': [layer] * len(target_corrs[layer])})
            
    dfs.append(df)
corr_results = pd.concat(dfs)

# melt fg_corrs and bg_corrs into one column
corr_results = corr_results.melt(id_vars=['layer'], value_vars=['fg_corrs', 'bg_corrs'], var_name='condition', value_name='correlation')

: 

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt


: 

: 

: 